<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/grid_image_splitter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
%%capture
!pip install -q opencv-python-headless Pillow numpy scipy matplotlib

In [ ]:
#@title ##**Upload** { display-mode: "form" }

from google.colab import files
from IPython.display import display, Image as IPImage
import os

print("📁 Select an image with a grid (jpg, png, webp):")
uploaded = files.upload()

IMAGE_PATH = list(uploaded.keys())[0]
print(f"✅ Uploaded: {IMAGE_PATH}")
display(IPImage(IMAGE_PATH, width=600))

In [ ]:
#@title ##**Run** { display-mode: "form" }
MODE = "Auto" #@param ["Auto", "Manual"]
#@markdown ---
#@markdown ### Manual Mode Configuration
ROWS = 3 #@param {type:"slider", min:1, max:12, step:1}
COLS = 1 #@param {type:"slider", min:1, max:12, step:1}
#@markdown ---

TRIM_PAGE_MARGINS = "No" #@param ["Yes", "No"]
TRIM_BORDERS = "Yes" #@param ["Yes", "No"]

SHOW_LOGS = "No" #@param ["Yes", "No"]

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import zipfile, os
import itertools
from operator import itemgetter
from google.colab import files
from scipy.signal import find_peaks

OUTPUT_DIR = 'split_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_TRIM_PCT = 15
REMOVE_SMALL_CELLS_LIMIT = 10

def dprint(msg):
    if SHOW_LOGS == "Yes":
        print(msg)

img_pil = Image.open(IMAGE_PATH).convert('RGB')
img_arr = np.array(img_pil)
H, W    = img_arr.shape[:2]
dprint(f'Original image size: {W}x{H} px')

def trim_panel_borders(crop_img, dark_limit, std_threshold=12.0):
    h, w = crop_img.shape[:2]
    gray = cv2.cvtColor(crop_img, cv2.COLOR_RGB2GRAY)

    max_trim_w = int(w * (MAX_TRIM_PCT / 100.0))
    max_trim_h = int(h * (MAX_TRIM_PCT / 100.0))

    y1, y2 = 0, h
    x1, x2 = 0, w

    candidate_x1 = 0
    for x in range(max_trim_w):
        col = gray[:, x]
        col_std = np.std(col)
        col_mean = np.mean(col)
        col_max = np.max(col)
        is_gutter = (col_std < std_threshold) and (col_mean < dark_limit or col_mean > 180)
        if is_gutter:
            candidate_x1 = x + 1
        else:
            break
    x1 = candidate_x1

    candidate_x2 = w
    for x in range(w - 1, w - max_trim_w - 1, -1):
        col = gray[:, x]
        col_std = np.std(col)
        col_mean = np.mean(col)
        col_max = np.max(col)
        is_gutter = (col_std < std_threshold) and (col_mean < dark_limit or col_mean > 180)
        if is_gutter:
            candidate_x2 = x
        else:
            break
    x2 = candidate_x2

    candidate_y1 = 0
    for y in range(max_trim_h):
        row = gray[y, :]
        row_std = np.std(row)
        row_mean = np.mean(row)
        row_max = np.max(row)
        is_gutter = (row_std < std_threshold) and (row_mean < dark_limit or row_mean > 180)
        if is_gutter:
            candidate_y1 = y + 1
        else:
            break
    y1 = candidate_y1

    candidate_y2 = h
    for y in range(h - 1, h - max_trim_h - 1, -1):
        row = gray[y, :]
        row_std = np.std(row)
        row_mean = np.mean(row)
        row_max = np.max(row)
        is_gutter = (row_std < std_threshold) and (row_mean < dark_limit or row_mean > 180)
        if is_gutter:
            candidate_y2 = y
        else:
            break
    y2 = candidate_y2

    if (x2 - x1) > int(w * 0.5) and (y2 - y1) > int(h * 0.5):
        return x1, y1, x2, y2
    return 0, 0, w, h

def find_gutters_auto(gray, axis='v', dark_limit=40):
    size = gray.shape[1] if axis == 'v' else gray.shape[0]
    axis_name = 'Vertical' if axis == 'v' else 'Horizontal'

    if axis == 'v':
        line_std = np.std(gray, axis=0)
    else:
        line_std = np.std(gray, axis=1)

    window_size = max(3, int(size * 0.005))
    if window_size % 2 == 0:
        window_size += 1
    smoothed_std = np.convolve(line_std, np.ones(window_size)/window_size, mode='same')

    local_min = np.percentile(smoothed_std, 8)
    local_med = np.median(smoothed_std)
    auto_threshold = local_min + (local_med - local_min) * 0.18
    auto_threshold = max(8.0, min(25.0, auto_threshold))

    gutter_mask = smoothed_std < auto_threshold

    margin = max(40, int(size * 0.08))
    gutter_mask[:margin] = False
    gutter_mask[-margin:] = False

    gutter_indices = np.where(gutter_mask)[0]
    seams = []
    if len(gutter_indices) > 0:
        for _, g in itertools.groupby(enumerate(gutter_indices), lambda ix: ix[0] - ix[1]):
            group = list(map(itemgetter(1), g))
            if len(group) >= 2:
                group_stds = smoothed_std[group]
                best_idx = group[np.argmin(group_stds)]
                seams.append(best_idx)

    inverted_std = 255.0 - smoothed_std
    peaks, _ = find_peaks(inverted_std, distance=max(10, int(size * 0.04)), prominence=1.0)
    peaks = [p for p in peaks if margin < p < size - margin]

    dprint(f"  [{axis_name}] Analyzed potential split lines:")
    for p in peaks:
        val = smoothed_std[p]
        status = "ACCEPTED" if val < auto_threshold else f"REJECTED (std {val:.2f} >= threshold {auto_threshold:.2f})"
        dprint(f"    Line at {p}: {status}")

    return seams, auto_threshold

def detect_panels_canny(img_arr):
    H, W = img_arr.shape[:2]
    gray = cv2.cvtColor(img_arr, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 30, 150)
    k_size = max(5, int(W * 0.01))
    if k_size % 2 == 0:
        k_size += 1
    kernel = np.ones((k_size, k_size), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=1)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    min_w = int(W * 0.08)
    min_h = int(H * 0.08)
    min_area = min_w * min_h
    bboxes = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w >= min_w and h >= min_h and (w * h) >= min_area:
            inset_x = int(k_size * 0.4)
            inset_y = int(k_size * 0.4)
            x1 = max(0, x + inset_x)
            y1 = max(0, y + inset_y)
            x2 = min(W, x + w - inset_x)
            y2 = min(H, y + h - inset_y)
            if (x2 - x1) >= min_w and (y2 - y1) >= min_h:
                bboxes.append((x1, y1, x2, y2))
    bboxes = sorted(bboxes, key=lambda b: (b[1] // 30, b[0]))
    return bboxes

dprint('Analyzing image and calculating base parameters...')
gray_full = cv2.cvtColor(img_arr, cv2.COLOR_RGB2GRAY)
blurred = cv2.medianBlur(gray_full, 5)

img_p5 = np.percentile(gray_full, 5)
dark_limit = max(15, min(80, img_p5 + 25))

start_y = 0
end_y = H
start_x = 0
end_x = W

if TRIM_PAGE_MARGINS == "Yes":
    global_std_h = np.std(gray_full, axis=1)
    global_std_w = np.std(gray_full, axis=0)
    global_gutter_thresh = max(10.0, np.percentile(global_std_h, 5) * 1.5)

    max_crop_h = int(H * 0.05)
    max_crop_w = int(W * 0.05)

    while start_y < max_crop_h and global_std_h[start_y] < global_gutter_thresh:
        start_y += 1
    while end_y > H - max_crop_h and global_std_h[end_y - 1] < global_gutter_thresh:
        end_y -= 1
    while start_x < max_crop_w and global_std_w[start_x] < global_gutter_thresh:
        start_x += 1
    while end_x > W - max_crop_w and global_std_w[end_x - 1] < global_gutter_thresh:
        end_x -= 1
    dprint(f"[LOG] Automatically cropped external page margins: Top/Bottom: {start_y}/{H-end_y}px, Left/Right: {start_x}/{W-end_x}px")
else:
    dprint("[LOG] Page margin trimming is disabled (TRIM_PAGE_MARGINS = No).")

img_arr = img_arr[start_y:end_y, start_x:end_x]
H, W = img_arr.shape[:2]
gray_full = gray_full[start_y:end_y, start_x:end_x]
blurred = blurred[start_y:end_y, start_x:end_x]

v_seams_log, auto_thresh_v = find_gutters_auto(blurred, 'v', dark_limit)
h_seams_log, auto_thresh_h = find_gutters_auto(blurred, 'h', dark_limit)

cells = []

if MODE == "Manual":
    dprint(f"[MANUAL MODE] Slicing image into {ROWS}x{COLS} grid...")
    v_intervals = [(int(j * W / COLS), int((j+1) * W / COLS)) for j in range(COLS)]
    h_intervals = [(int(i * H / ROWS), int((i+1) * H / ROWS)) for i in range(ROWS)]

    plot_h_seams = [y for y, _ in h_intervals[1:]]
    plot_v_seams = [x for x, _ in v_intervals[1:]]

    for i, (y1, y2) in enumerate(h_intervals):
        for j, (x1, x2) in enumerate(v_intervals):
            crop = img_arr[y1:y2, x1:x2]
            if TRIM_BORDERS == "Yes":
                tx1, ty1, tx2, ty2 = trim_panel_borders(crop, dark_limit, std_threshold=12.0)
            else:
                tx1, ty1, tx2, ty2 = 0, 0, crop.shape[1], crop.shape[0]

            final_x1 = x1 + tx1
            final_y1 = y1 + ty1
            final_x2 = x1 + tx2
            final_y2 = y1 + ty2
            trimmed_crop = img_arr[final_y1:final_y2, final_x1:final_x2]

            cells.append({
                'img': trimmed_crop,
                'raw_bbox': (x1, y1, x2, y2),
                'bbox': (final_x1, final_y1, final_x2, final_y2),
                'row': i,
                'col': j,
                'trim_values': (tx1, crop.shape[1]-tx2, ty1, crop.shape[0]-ty2)
            })
else:
    dprint("[AUTO MODE] Analysing layout dynamically...")
    plot_h_seams = h_seams_log
    plot_v_seams = v_seams_log

    detected_bboxes = detect_panels_canny(img_arr)

    grid_cells_count = (len(h_seams_log) + 1) * (len(v_seams_log) + 1)
    canny_cells_count = len(detected_bboxes)

    use_canny = False
    if canny_cells_count > 1:
        if grid_cells_count >= canny_cells_count and (len(h_seams_log) > 0 or len(v_seams_log) > 0):
            dprint(f"  [LOG] Auto-Grid found more structured layout ({grid_cells_count} panels) than Canny ({canny_cells_count}). Prioritizing grid.")
            use_canny = False
        else:
            dprint(f"  [LOG] Irregular layout detected. Using dynamic Canny segmentation ({canny_cells_count} panels).")
            use_canny = True
    else:
        use_canny = False

    if use_canny:
        for idx, bbox in enumerate(detected_bboxes):
            x1, y1, x2, y2 = bbox
            crop = img_arr[y1:y2, x1:x2]
            if TRIM_BORDERS == "Yes":
                tx1, ty1, tx2, ty2 = trim_panel_borders(crop, dark_limit, std_threshold=12.0)
            else:
                tx1, ty1, tx2, ty2 = 0, 0, crop.shape[1], crop.shape[0]

            final_x1 = x1 + tx1
            final_y1 = y1 + ty1
            final_x2 = x1 + tx2
            final_y2 = y1 + ty2
            trimmed_crop = img_arr[final_y1:final_y2, final_x1:final_x2]

            cells.append({
                'img': trimmed_crop,
                'raw_bbox': (x1, y1, x2, y2),
                'bbox': (final_x1, final_y1, final_x2, final_y2),
                'row': idx,
                'col': 0,
                'trim_values': (tx1, crop.shape[1]-tx2, ty1, crop.shape[0]-ty2)
            })
    else:
        h_bounds = [0] + h_seams_log + [H]
        v_bounds = [0] + v_seams_log + [W]

        for i in range(len(h_bounds)-1):
            for j in range(len(v_bounds)-1):
                y1, y2 = h_bounds[i], h_bounds[i+1]
                x1, x2 = v_bounds[j], v_bounds[j+1]
                crop = img_arr[y1:y2, x1:x2]
                if TRIM_BORDERS == "Yes":
                    tx1, ty1, tx2, ty2 = trim_panel_borders(crop, dark_limit, std_threshold=12.0)
                else:
                    tx1, ty1, tx2, ty2 = 0, 0, crop.shape[1], crop.shape[0]

                final_x1 = x1 + tx1
                final_y1 = y1 + ty1
                final_x2 = x1 + tx2
                final_y2 = y1 + ty2
                trimmed_crop = img_arr[final_y1:final_y2, final_x1:final_x2]

                cells.append({
                    'img': trimmed_crop,
                    'raw_bbox': (x1, y1, x2, y2),
                    'bbox': (final_x1, final_y1, final_x2, final_y2),
                    'row': i,
                    'col': j,
                    'trim_values': (tx1, crop.shape[1]-tx2, ty1, crop.shape[0]-ty2)
                })

if TRIM_BORDERS == "Yes" and len(cells) > 1:
    dprint("[LOG] Running panel alignment post-processing...")

    w_groups = {}
    for c in cells:
        rx1, ry1, rx2, ry2 = c['raw_bbox']
        raw_w = rx2 - rx1
        if raw_w not in w_groups:
            w_groups[raw_w] = []
        w_groups[raw_w].append(c)

    for raw_w, group in w_groups.items():
        if len(group) > 1:
            min_trim_left = min(c['trim_values'][0] for c in group)
            min_trim_right = min(c['trim_values'][1] for c in group)
            dprint(f"  Width group (raw_w={raw_w}): keeping minimal trims: Left={min_trim_left}px, Right={min_trim_right}px")
            for c in group:
                rx1, ry1, rx2, ry2 = c['raw_bbox']
                tx1, tx_r, ty1, ty_b = c['trim_values']
                new_tx1 = min(tx1, min_trim_left)
                new_tx_r = min(tx_r, min_trim_right)
                c['bbox'] = (rx1 + new_tx1, c['bbox'][1], rx2 - new_tx_r, c['bbox'][3])
                c['trim_values'] = (new_tx1, new_tx_r, ty1, ty_b)

    h_groups = {}
    for c in cells:
        rx1, ry1, rx2, ry2 = c['raw_bbox']
        raw_h = ry2 - ry1
        if raw_h not in h_groups:
            h_groups[raw_h] = []
        h_groups[raw_h].append(c)

    for raw_h, group in h_groups.items():
        if len(group) > 1:
            min_trim_top = min(c['trim_values'][2] for c in group)
            min_trim_bottom = min(c['trim_values'][3] for c in group)
            dprint(f"  Height group (raw_h={raw_h}): keeping minimal trims: Top={min_trim_top}px, Bottom={min_trim_bottom}px")
            for c in group:
                rx1, ry1, rx2, ry2 = c['raw_bbox']
                tx1, tx_r, ty1, ty_b = c['trim_values']
                new_ty1 = min(ty1, min_trim_top)
                new_ty_b = min(ty_b, min_trim_bottom)
                c['bbox'] = (c['bbox'][0], ry1 + new_ty1, c['bbox'][2], ry2 - new_ty_b)
                c['trim_values'] = (tx1, tx_r, new_ty1, new_ty_b)

    for c in cells:
        x1, y1, x2, y2 = c['bbox']
        c['img'] = img_arr[y1:y2, x1:x2]

if SHOW_LOGS == "Yes":
    print("\n[LOG] Detailed cell-by-cell trim results:")
    for i, c in enumerate(cells):
        tx_l, tx_r, ty_t, ty_b = c['trim_values']
        print(f"  Cell #{i+1:02d}:")
        print(f"    Raw split bbox    : {c['raw_bbox']}")
        print(f"    Trimmed border    : Left={tx_l}px, Right={tx_r}px, Top={ty_t}px, Bottom={ty_b}px")
        print(f"    Final cropped bbox: {c['bbox']}")

if len(cells) > 0:
    max_w = max(c['img'].shape[1] for c in cells)
    max_h = max(c['img'].shape[0] for c in cells)
    filtered_cells = []
    for c in cells:
        cell_h, cell_w = c['img'].shape[:2]
        if cell_w < (max_w / float(REMOVE_SMALL_CELLS_LIMIT)) or cell_h < (max_h / float(REMOVE_SMALL_CELLS_LIMIT)):
            continue
        filtered_cells.append(c)
    cells = filtered_cells

print(f'\nTotal panels successfully split: {len(cells)}')

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(img_arr)
ax.set_title(f'Split Results ({MODE} Mode): {len(cells)} cells (Dashed Red = Grid, Solid Blue = Final Output)', fontsize=13)

for h in plot_h_seams:
    ax.axhline(y=h, color='lime', linewidth=1.5, linestyle=':')
for vl in plot_v_seams:
    ax.axvline(x=vl, color='red', linewidth=1.5, linestyle=':')

for idx, c in enumerate(cells):
    rx1, ry1, rx2, ry2 = c['raw_bbox']
    x1, y1, x2, y2 = c['bbox']

    rect_raw = patches.Rectangle((rx1, ry1), rx2-rx1, ry2-ry1, linewidth=1.5, edgecolor='red', linestyle='--', facecolor='none')
    ax.add_patch(rect_raw)

    rect_final = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='blue', facecolor='none')
    ax.add_patch(rect_final)

    ax.text((x1+x2)//2, (y1+y2)//2, f"#{idx+1}",
            ha='center', va='center', color='white', fontsize=13, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.6))
ax.axis('off')
plt.tight_layout()
plt.savefig('grid_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

n = len(cells)
if n > 0:
    cols_p = min(n, 5)
    rows_p = (n + cols_p - 1) // cols_p
    fig2, axes2 = plt.subplots(rows_p, cols_p, figsize=(cols_p*3, rows_p*3))
    ax_flat = np.array(axes2).flatten()
    for idx, cell in enumerate(cells):
        ax_flat[idx].imshow(cell['img'])
        sh = cell['img'].shape
        ax_flat[idx].set_title(f'#{idx+1}  {sh[1]}x{sh[0]}', fontsize=8)
        ax_flat[idx].axis('off')
    for idx in range(n, len(ax_flat)):
        ax_flat[idx].axis('off')
    plt.suptitle('Processed Panels', fontsize=14)
    plt.tight_layout()
    plt.show()

if len(cells) <= 1:
    print("\n⚠ Output has only 1 cell. ZIP download skipped.")
else:
    base = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
    zname = 'split_images.zip'
    with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as zf:
        for idx, cell in enumerate(cells):
            fname = f'{base}_{idx+1}.png'
            fpath = os.path.join(OUTPUT_DIR, fname)
            Image.fromarray(cell['img']).save(fpath)
            zf.write(fpath, fname)
            sh = cell['img'].shape
            print(f'  #{idx+1:02d}  {sh[1]}x{sh[0]}  {fname}')

    print(f'\nDone! Downloading archive {zname} with {len(cells)} parts...')
    files.download(zname)